# AnnData, `.h5ad`, and pseudobulk for this challenge

This notebook is for **competitors** who want to understand:

1. **AnnData** and the **`.h5ad`** file format used across single-cell tooling (Scanpy, scVI, etc.).
2. How this repository builds **pseudobulk** from cell-level data for age prediction.

**Official AnnData documentation:** [anndata.readthedocs.io](https://anndata.readthedocs.io/en/latest/index.html) — the format sits between pandas and tensor-style arrays and is central to the [scverse](https://scverse.org/) ecosystem.

**Requirements:** `scanpy`, `pandas`, `numpy` (same environment as the baseline model tutorial).

## Part A — What is AnnData?

An **`AnnData`** object (`anndata.AnnData`) bundles:

| Slot | Role | Typical content |
|------|------|-----------------|
| **`X`** | Main data matrix | `n_obs × n_vars` (cells × genes), often sparse counts |
| **`obs`** | Row annotations | cell IDs, QC, `donor_id`, `celltype`, `age`, … |
| **`var`** | Column annotations | gene symbols / Ensembl IDs, optional flags |
| **`layers`** | Extra matrices | raw counts, normalized data, imputed |
| **`obsm` / `varm`** | Multi-column embeddings | PCA, UMAP coordinates |
| **`uns`** | Unstructured dict | pipeline parameters, colors, … |

**Naming:** `obs` = observations (here: **cells**, or later **donor×celltype** rows). `var` = variables (here: **genes**).

**.h5ad** is the HDF5-based on-disk format for AnnData (`read_h5ad` / `write_h5ad`). See the AnnData docs for [on-disk layout and interoperability](https://anndata.readthedocs.io/en/latest/index.html).

**Teaching point:** Inspect `adata.shape`, `adata.obs.head()`, and `adata.var_names` first whenever you open a new file.

### Visual schema

![AnnData schematic: matrix X with obs (rows) and var (columns) and linked annotation slots](../images/anndata_schema_image.svg)

Repository file: [`images/anndata_schema_image.svg`](../images/anndata_schema_image.svg) (relative to project root; from this notebook the path above is `../images/...`).

### A1 — Project path

Run from the repository root or from `notebooks/`.

In [ ]:
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
if (NOTEBOOK_DIR / "models" / "train_age_model.py").exists():
    PROJ_ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent / "models" / "train_age_model.py").exists():
    PROJ_ROOT = NOTEBOOK_DIR.parent
else:
    raise FileNotFoundError(
        "Could not find models/train_age_model.py. Open Jupyter from the repo root or from notebooks/."
    )

# Competition data live under `data/` (bind or copy from shared scratch — see README).
DATA_DIR = PROJ_ROOT / "data"
RESULTS_DIR = PROJ_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJ_ROOT)
print("Data directory:", DATA_DIR)
print("Results directory:", RESULTS_DIR)

Project root: /iridisfs/ddnb/Ahmed/aging_challenge_2026


### A2 — Explore competition **cell-level** `.h5ad` (lightweight peek)

Full objects (~1M+ cells) are **large in RAM**. Two strategies:

1. **`backed='r'`** — keep `X` on disk while you inspect `obs` / `var` (see [AnnData backed mode](https://anndata.readthedocs.io/en/latest/generated/anndata.AnnData.html)).
2. **Pseudobulk outputs** (next sections) — much smaller; use those for quick plots.

Files are read from **`data/`** (e.g. `data/combined_public.h5ad`). Adjust `CELL_LEVEL` only if you use a custom layout.

In [5]:
import scanpy as sc

CELL_LEVEL = DATA_DIR / "combined_public.h5ad"

if not CELL_LEVEL.exists():
    print("No cell-level file at:", CELL_LEVEL)
    print("Copy or bind competition data into data/ (see README), or skip to pseudobulk sections.")
else:
    ad_cell = sc.read_h5ad(CELL_LEVEL, backed="r")
    try:
        print("Shape (obs × var):", ad_cell.n_obs, "×", ad_cell.n_vars)
        print("obs columns (sample):", list(ad_cell.obs.columns[:12]), "...")
        print(ad_cell.obs[["donor_id", "celltype", "age", "_split"]].head(6))
    finally:
        if getattr(ad_cell, "isbacked", False) and ad_cell.file is not None:
            ad_cell.file.close()

Shape (obs × var): 1183459 × 28278
obs columns (sample): ['orig.ident', 'n_genes_by_counts', 'pct_counts_mt', 'donor_id', 'pool_number', 'age', 'celltype', '_split'] ...
                   donor_id celltype   age _split
barcode                                          
AAACCTGAGAATGTTG-1      677    CD4 T  66.0  train
AAACCTGAGTATTGGA-1      668    CD8 T  78.0  train
AAACCTGAGTGTCCCG-1      669    CD8 T  75.0  train
AAACCTGCAAGACGTG-1      674    CD4 T  59.0  train
AAACCTGCAGCATACT-1      676    CD8 T  68.0  train
AAACCTGCAGCATGAG-1      677    CD8 T  66.0  train


**Competition columns (cell-level `obs`):** `donor_id`, `celltype` (5 coarse groups), `age` (missing on test cells in some builds), `_split` (`train` / `val` / `test`), plus QC fields — see the main `README.md`.

Each **row** is one cell; `X[i, g]` is the expression of gene `g` in cell `i`.

### A3 — Explore **pseudobulk combined** `.h5ad` (recommended demo size)

After `h5ad_to_pseudobulk.py` on cell-level `combined*.h5ad`, you get:

- **`combined_pseudobulk_combined_public.h5ad`** (public release): rows = **(donor, cell type)** pairs; columns = genes; values = **sum of counts** over all cells in that donor and type.

Dimensions are roughly **4,902 × 28,278** (981 donors × 5 cell types × genes) — feasible to load fully in memory for exploration.

In [6]:
import numpy as np
import pandas as pd
import scanpy as sc
from IPython.display import display

PB_COMBINED = DATA_DIR / "pseudobulk" / "combined_pseudobulk_combined_public.h5ad"

if not PB_COMBINED.exists():
    print("No file at:", PB_COMBINED)
    print("Run: python data_prep/h5ad_to_pseudobulk.py data/combined_public.h5ad -o data/pseudobulk")
else:
    ad_pb = sc.read_h5ad(PB_COMBINED)
    print("Shape:", ad_pb.n_obs, "rows ×", ad_pb.n_vars, "genes")
    print("obs columns:", list(ad_pb.obs.columns))
    display(ad_pb.obs[["donor_id", "celltype", "age", "_split", "n_cells"]].head(10))
    # Gene expression lives in X (var dimension), not as columns in obs
    g0 = ad_pb.var_names[0]
    print(f"Example gene {g0!r} — first rows of pseudobulk counts (from .X):")
    Xcol = ad_pb[:10, g0].X
    if hasattr(Xcol, "toarray"):
        Xcol = Xcol.toarray()
    counts = np.asarray(Xcol).ravel()
    demo_g = ad_pb.obs[["donor_id", "celltype"]].head(10).copy()
    demo_g[g0] = counts
    display(demo_g)

Shape: 4902 rows × 28278 genes
obs columns: ['donor_id', 'orig.ident', 'n_genes_by_counts', 'pct_counts_mt', 'pool_number', 'age', '_split', 'celltype', 'n_cells', 'n_cells_CD4_T', 'n_cells_CD8_T', 'n_cells_NK', 'n_cells_B_cells', 'n_cells_monocytes']


,donor_id,celltype,age,_split,n_cells
677_CD4 T,677,CD4 T,66.0,train,874
674_CD4 T,674,CD4 T,59.0,train,1040
668_CD4 T,668,CD4 T,78.0,train,669
669_CD4 T,669,CD4 T,75.0,train,375
676_CD4 T,676,CD4 T,68.0,train,542
670_CD4 T,670,CD4 T,75.0,train,457
805_CD4 T,805,CD4 T,75.0,train,1001
807_CD4 T,807,CD4 T,65.0,train,1270
814_CD4 T,814,CD4 T,71.0,train,1407
806_CD4 T,806,CD4 T,66.0,train,697


Example gene 'ENSG00000120162' — first rows of pseudobulk counts (from .X):


,donor_id,celltype,ENSG00000120162
677_CD4 T,677,CD4 T,0.0
674_CD4 T,674,CD4 T,4.0
668_CD4 T,668,CD4 T,0.0
669_CD4 T,669,CD4 T,0.0
676_CD4 T,676,CD4 T,1.0
670_CD4 T,670,CD4 T,0.0
805_CD4 T,805,CD4 T,2.0
807_CD4 T,807,CD4 T,3.0
814_CD4 T,814,CD4 T,2.0
806_CD4 T,806,CD4 T,0.0


### A4 — **Donor-aggregated** matrix (what the baseline model reads)

`combined_pseudobulk_donor_aggregated_public.h5ad` has **one row per donor** and **one column per (gene, cell-type slot)** with names like `ENSG...__CD4_T`.

So each gene contributes **5 numbers** per donor (CD4 T, CD8 T, NK, B cells, monocytes). Total columns ≈ `5 × n_genes`.

In [7]:
PB_AGG = DATA_DIR / "pseudobulk" / "combined_pseudobulk_donor_aggregated_public.h5ad"

if not PB_AGG.exists():
    print("No donor-aggregated file at:", PB_AGG)
else:
    ad_agg = sc.read_h5ad(PB_AGG)
    print("Shape:", ad_agg.n_obs, "donors ×", ad_agg.n_vars, "features")
    print("Example var_names:", list(ad_agg.var_names[:6]))
    display(ad_agg.obs[["donor_id", "age", "_split"]].head())

Shape: 981 donors × 141390 features
Example var_names: ['ENSG00000120162__CD4_T', 'ENSG00000120162__CD8_T', 'ENSG00000120162__NK', 'ENSG00000120162__B_cells', 'ENSG00000120162__monocytes', 'ENSG00000134668__CD4_T']


,donor_id,age,_split
1,1,73.0,train
10,10,72.0,test
100,100,50.0,val
101,101,71.0,train
102,102,74.0,train


## Part B — Pseudobulk method (this repository)

### Step 1 — Sum counts per **(donor, cell type)**

From cell-level `AnnData`, for each cell type group we compute:

$$ \text{pseudobulk}_{d,c,g} = \sum_{\text{cells } i \text{ with donor } d,\, \text{type } c} X_{i,g} $$

Implementation: `h5ad_to_pseudobulk()` in `data_prep/h5ad_to_pseudobulk.py` loops over cell types, masks cells, then sums each donor’s rows in `X` (or a chosen `layer`). Donor metadata is attached from the first matching cell; **`n_cells`** stores how many cells were summed per donor and type.

Per–cell-type objects are also written as `combined_CD4_T.h5ad`, … if you run the full script.

### Step 2 — Stack all types → **combined** `(donor, celltype) × genes`

`scanpy.concat(..., axis=0)` stacks the per-type `AnnData` objects so each row is still one donor in one cell type.

### Step 3 — **Donor-level wide matrix** for ML

`aggregate_to_donor_level()` reshapes to **one row per donor** and fills blocks of columns in fixed order: CD4 T, CD8 T, NK, B cells, monocytes — so feature `GENE__CD4_T` is the pseudobulk count of that gene in CD4 T cells for that donor, etc.

**Why pseudobulk?** It collapses millions of cells into **interpretable donor-level features**, reduces noise from individual dropout events, and matches common practice when the prediction target is **donor metadata** (here: age) rather than single-cell labels.

**Caveat:** Summing raw counts assumes counts are comparable within cell type; depth differences across donors still matter (competitors may normalise or use offsets — the baseline applies `log1p` before the Random Forest).

### B1 — **Minimal synthetic example** (always runs)

We build a toy `AnnData` with 2 donors × 5 cell types × a few genes and call the **same functions** as the pipeline so you see the shapes change: cells → pseudobulk by type → concatenated → donor-aggregated.

In [8]:
import sys

sys.path.insert(0, str(PROJ_ROOT / "data_prep"))
import h5ad_to_pseudobulk as pb  # noqa: E402

np.random.seed(0)
cts = ["CD4 T", "CD8 T", "NK", "B cells", "monocytes"]
genes = ["GENE_A", "GENE_B", "GENE_C"]
rows = []
X_rows = []
# Two donors; each donor has 3 cells per cell type (30 rows total)
for donor in ["101", "102"]:
    base = 20 if donor == "101" else 35
    for ct in cts:
        for _cell in range(3):
            rows.append({"donor_id": donor, "celltype": ct, "age": float(base), "_split": "train"})
            noise = np.random.poisson(1, size=len(genes))
            signal = np.array([5, 2, 1]) * (cts.index(ct) + 1)
            X_rows.append(signal + noise)

obs_toy = pd.DataFrame(rows)
X_toy = np.asarray(X_rows, dtype=np.float64)
var_toy = pd.DataFrame(index=genes)
adata_toy = sc.AnnData(X=X_toy, obs=obs_toy, var=var_toy)

print("Cell-level:", adata_toy.n_obs, "cells ×", adata_toy.n_vars, "genes")
by_ct = pb.h5ad_to_pseudobulk(adata_toy, donor_col="donor_id", celltype_col="celltype")
for ct, sub in by_ct.items():
    print(f"  {ct}: {sub.n_obs} donors × {sub.n_vars} genes (sums), n_cells col =", sub.obs["n_cells"].tolist())

ad_stack = sc.concat(list(by_ct.values()), axis=0, join="inner")
print("Stacked:", ad_stack.n_obs, "(donor×type) rows ×", ad_stack.n_vars, "genes")

ad_wide = pb.aggregate_to_donor_level(ad_stack, donor_col="donor_id", celltype_col="celltype")
print("Donor-aggregated:", ad_wide.n_obs, "donors ×", ad_wide.n_vars, "features")
print("Feature names (3 genes × 5 types = 15):", list(ad_wide.var_names))

Cell-level: 30 cells × 3 genes
  CD4 T: 2 donors × 3 genes (sums), n_cells col = [3, 3]
  CD8 T: 2 donors × 3 genes (sums), n_cells col = [3, 3]
  NK: 2 donors × 3 genes (sums), n_cells col = [3, 3]
  B cells: 2 donors × 3 genes (sums), n_cells col = [3, 3]
  monocytes: 2 donors × 3 genes (sums), n_cells col = [3, 3]
Stacked: 10 (donor×type) rows × 3 genes
Donor-aggregated: 2 donors × 15 features
Feature names (3 genes × 5 types = 15): ['GENE_A__CD4_T', 'GENE_A__CD8_T', 'GENE_A__NK', 'GENE_A__B_cells', 'GENE_A__monocytes', 'GENE_B__CD4_T', 'GENE_B__CD8_T', 'GENE_B__NK', 'GENE_B__B_cells', 'GENE_B__monocytes', 'GENE_C__CD4_T', 'GENE_C__CD8_T', 'GENE_C__NK', 'GENE_C__B_cells', 'GENE_C__monocytes']


### B2 — Reproduce pipeline command (optional)

From the project root, after `data/combined_public.h5ad` exists:

```bash
python data_prep/h5ad_to_pseudobulk.py data/combined_public.h5ad -o data/pseudobulk
```

Or via your Apptainer wrapper / SLURM script pointing at that Python module.

***

**Further reading:** [AnnData API](https://anndata.readthedocs.io/en/latest/api.html), project `README.md` (Pseudobulk section), and `data_prep/h5ad_to_pseudobulk.py` source.